# Churn Embedding Model — Ray + Kubeflow Demo

This notebook walks through the full demo:
1. Apply Kubernetes manifests (Ray cluster + secrets)
2. Verify the Ray cluster is healthy
3. Generate synthetic data and upload to S3
4. Compile and submit the Kubeflow Pipeline
5. Monitor training via Ray dashboard
6. Inspect results

## 0. Install dependencies

In [ ]:
!pip install -q kfp==2.7.0 ray[train]==2.9.0 torch boto3 pandas numpy pyarrow scikit-learn

## 1. Deploy Ray Cluster to Kubernetes

In [ ]:
# Edit your S3 credentials in s3_secret.yaml and bucket/endpoint in ray_cluster.yaml before running
!kubectl apply -f s3_secret.yaml
!kubectl apply -f ray_cluster.yaml
!kubectl apply -f ray_service.yaml

In [ ]:
# Wait for both workers to be Ready (takes ~60-90s on first pull)
!kubectl wait --for=condition=ready pod \
    -l ray.io/cluster=churn-ray-cluster \
    -n kubeflow \
    --timeout=300s

## 2. Verify Ray Cluster

In [ ]:
import ray

RAY_ADDRESS = "ray://churn-ray-cluster-head-svc.kubeflow.svc.cluster.local:10001"

ray.init(address=RAY_ADDRESS, ignore_reinit_error=True)
print(ray.cluster_resources())
# Expected: {'CPU': 16.0, 'GPU': 2.0, 'memory': ...}
# Confirms 2 workers each with 1 GPU and 8 CPUs

In [ ]:
# List all nodes in the cluster
for node in ray.nodes():
    alive  = node['Alive']
    res    = node['Resources']
    addr   = node['NodeManagerAddress']
    gpu    = res.get('GPU', 0)
    cpu    = res.get('CPU', 0)
    print(f"  {'✅' if alive else '❌'}  {addr:<18}  CPU={cpu:.0f}  GPU={gpu:.0f}")

## 3. Generate Synthetic Data → S3
Run the data generator directly from the notebook to confirm S3 connectivity before submitting the pipeline.

In [ ]:
import sys
from pathlib import Path

# Assume the notebook runs with CWD = churn/ (same directory as the scripts).
_root = Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from generate_data import generate_churn_dataset, upload_to_s3
from s3_shakeout import load_config

S3_DATA_KEY  = "churn/train.parquet"
S3_MODEL_KEY = "models/churn_embedding_model.pt"

_cfg = load_config()
df = generate_churn_dataset(n=2000)
upload_to_s3(df, bucket=_cfg["bucket"], key=S3_DATA_KEY)
df.head()

In [ ]:
# Sanity check — churners should have higher tickets, lower spend
df.groupby('churned')[['tenure_months','support_tickets','monthly_spend']].mean().round(2)

## 4. (Optional) Run Training Directly via Ray
Skip straight to Ray training without the full pipeline — useful for rapid iteration.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from train import run_training

result = run_training(
    ray_address  = RAY_ADDRESS,
    s3_data_key  = S3_DATA_KEY,
    epochs       = 10,
    batch_size   = 256,
    num_workers  = 2,   # matches our 2-node cluster
)

print("\nFinal metrics:")
print(result.metrics)

## 5. Compile Kubeflow Pipeline

In [ ]:
import subprocess
import sys
from pathlib import Path

churn_dir = Path.cwd()
result = subprocess.run(
    [sys.executable, str(churn_dir / "churn_pipeline.py")],
    capture_output=True,
    text=True,
    cwd=str(churn_dir),
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

## 6. Submit Pipeline to Kubeflow

In [ ]:
import kfp

KFP_HOST = 'http://ml-pipeline.kubeflow.svc.cluster.local:8888'  # adjust if needed

client = kfp.Client(host=KFP_HOST)
print('KFP version:', client.get_kfp_healthz())

In [ ]:
from pathlib import Path

run = client.create_run_from_pipeline_package(
    pipeline_file=str(Path.cwd() / "churn_pipeline.yaml"),
    arguments={
        "ray_address":   RAY_ADDRESS,
        # Empty string: KFP steps use AWS_S3_BUCKET from their pod env (same as generate_data.py).
        "s3_bucket":     "",
        "s3_data_key":   S3_DATA_KEY,
        "s3_model_key":  S3_MODEL_KEY,
        "n_samples":     2000,
        "num_workers":   2,
        "epochs":        10,
        "auc_threshold": 0.70,
    },
    run_name="churn-ray-demo-run",
    experiment_name="churn-embedding",
)

print(f'Run ID  : {run.run_id}')
print(f'Run URL : {KFP_HOST}/#/runs/details/{run.run_id}')

## 7. Monitor the Pipeline Run

In [ ]:
import time
from IPython.display import clear_output

# Poll every 15s and print current step status
while True:
    run_response = client.get_run(run.run_id)
    status       = run_response.state
    clear_output(wait=True)
    print(f'Pipeline status: {status}')

    for node in (run_response.pipeline_spec or {}).get('tasks', []):
        print(f"  {node.get('taskInfo',{}).get('name','?'):<30} {node.get('state','?')}")

    if status in ('SUCCEEDED', 'FAILED', 'CANCELED'):
        break
    time.sleep(15)

## 8. Monitor Ray Dashboard
Port-forward the Ray dashboard to your local machine to see live GPU utilisation and worker task assignment.

In [ ]:
# Run this in a terminal (outside the notebook):
# kubectl port-forward svc/churn-ray-cluster-head-svc 8265:8265 -n kubeflow
# Then open: http://localhost:8265

# Or launch from the notebook (backgrounded):
import subprocess
subprocess.Popen([
    'kubectl', 'port-forward',
    'svc/churn-ray-cluster-head-svc',
    '8265:8265', '-n', 'kubeflow'
])
print('Ray dashboard → http://localhost:8265')

## 9. Inspect Results

In [ ]:
# Pull the saved model from S3 and run a quick local inference check
import io, sys, torch, numpy as np, pandas as pd
from pathlib import Path

_root = Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from s3_shakeout import load_config, make_s3_client
from train import ChurnEmbeddingModel, ChurnDataset
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, classification_report

_cfg = load_config()
s3 = make_s3_client(_cfg)
obj = s3.get_object(Bucket=_cfg["bucket"], Key=S3_MODEL_KEY)
state_dict = torch.load(io.BytesIO(obj["Body"].read()), map_location="cpu")

model = ChurnEmbeddingModel()
model.load_state_dict(state_dict)
model.eval()
print('Model loaded ✅')

In [ ]:
# Score on the held-out 20%
obj = s3.get_object(Bucket=_cfg["bucket"], Key=S3_DATA_KEY)
df  = pd.read_parquet(io.BytesIO(obj['Body'].read()))
df_test = df.iloc[int(len(df)*0.8):]

PLAN_MAP   = {'basic': 0, 'pro': 1, 'enterprise': 2}
REGION_MAP = {'north': 0, 'south': 1, 'east': 2, 'west': 3}

cat    = torch.tensor(np.stack([df_test['plan_type'].map(PLAN_MAP).values,
                                 df_test['region'].map(REGION_MAP).values], 1), dtype=torch.long)
num    = torch.tensor(df_test[['tenure_months','monthly_spend','support_tickets']].values, dtype=torch.float32)
seq    = torch.tensor(np.stack(df_test['usage_seq'].values), dtype=torch.float32).unsqueeze(-1)
labels = df_test['churned'].values

with torch.no_grad():
    preds = model(cat, num, seq).numpy()

print(f'AUC-ROC : {roc_auc_score(labels, preds):.4f}')
print()
print(classification_report(labels, (preds > 0.5).astype(int),
                             target_names=['Retained','Churned']))

In [ ]:
# Visualise churn score distribution
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(preds[labels==0], bins=40, alpha=0.6, label='Retained', color='steelblue')
ax.hist(preds[labels==1], bins=40, alpha=0.6, label='Churned',  color='tomato')
ax.axvline(0.5, color='black', linestyle='--', label='Threshold 0.5')
ax.set_xlabel('Predicted churn probability')
ax.set_ylabel('Count')
ax.set_title('Churn Score Distribution — Held-out Test Set')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Teardown (optional)

In [ ]:
# Uncomment to delete the Ray cluster when done
# !kubectl delete -f ../k8s/ray_cluster.yaml
# !kubectl delete -f ../k8s/ray_service.yaml
# ray.shutdown()